# ALSApp - Cattle disease classifier (on-device)

Trains a small MobileNetV2-based classifier on a public Kaggle cattle-disease
dataset (healthy / lumpy skin disease / foot-and-mouth disease, ~3.2k photos)
and exports it as a `.tflite` file to ship inside the Flutter app.

**v1 scope note:** this only covers cattle, and only these 3 classes. Sheep/goat/
poultry/bee disease datasets are too sparse right now to include - a later pass
can add them the same way once better public data exists.

**Before running:**
1. `Runtime` -> `Change runtime type` -> select a `T4 GPU`.
2. Get a free Kaggle API token: kaggle.com -> your avatar -> `Settings` -> `API` ->
   `Create New Token`. Open it and copy the `username` and `key` values.
3. In this Colab, click the key icon (🔑) in the left sidebar ("Secrets") and add two
   secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY`, with notebook access enabled for both.

**At the end** this notebook downloads two files - save them into
`frontend/assets/models/` in the app repo:
- `cattle_disease.tflite`
- `cattle_labels.txt`

In [ ]:
!pip install -q -U tensorflow scikit-learn kagglehub

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kagglehub

# Primary source: a single dataset already split into exactly the 3 classes
# we want (healthy / lumpy skin disease / foot-and-mouth disease, ~3.2k images).
dataset_path = kagglehub.dataset_download('devang03mgr/cattle-diseases-datasets')
print('Downloaded to:', dataset_path)

In [ ]:
# Kaggle dataset folder layouts vary and aren't documented in a machine-
# readable way - print the tree so you can SEE the real structure. If the
# auto-detection in the next cell picks the wrong folder, set DATA_DIR below
# manually based on what you see printed here (it should be the folder that
# directly contains one subfolder per class).
!find "{dataset_path}" -maxdepth 4 | head -60

In [ ]:
def find_class_dir(root):
    """Walk down while there's exactly one subdirectory (common Kaggle
    zip-wrapper pattern), stop at the first level that has 2+ subdirectories
    - that's almost certainly the folder-per-class root."""
    current = root
    for _ in range(6):
        subdirs = [d for d in os.scandir(current) if d.is_dir()]
        if len(subdirs) != 1:
            return current
        current = subdirs[0].path
    return current

DATA_DIR = find_class_dir(dataset_path)  # override manually here if needed
print('Using DATA_DIR =', DATA_DIR)
print('Detected class folders:', [d.name for d in os.scandir(DATA_DIR) if d.is_dir()])

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.15, subset='training', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.15, subset='validation', seed=42,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)

class_names = train_ds.class_names
num_classes = len(class_names)
print(num_classes, 'classes:', class_names)

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomBrightness(0.15),
    tf.keras.layers.RandomContrast(0.15),
])

train_ds = train_ds.map(lambda x, y: (augment(x, training=True), y),
                         num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
# Same recipe as the plant classifier - rescaling baked into the model so
# Flutter just feeds plain 0-255 RGB pixels.
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = tf.keras.layers.Rescaling(1. / 127.5, offset=-1)(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Small dataset (~3k images) - more epochs than the plant model, still fast.
model.fit(train_ds, validation_data=val_ds, epochs=12)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=6)

In [ ]:
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
print(
    'With only ~3k source images, watch the per-class numbers above, not just '
    'overall accuracy - the smallest class (foot-and-mouth, ~23% of the data) '
    'is the one most likely to need more data later.'
)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('cattle_disease.tflite', 'wb') as f:
    f.write(tflite_model)
with open('cattle_labels.txt', 'w') as f:
    f.write('\n'.join(class_names))

print(f"cattle_disease.tflite: {os.path.getsize('cattle_disease.tflite') / 1e6:.1f} MB")

In [ ]:
from google.colab import files
files.download('cattle_disease.tflite')
files.download('cattle_labels.txt')

## Improving this later

If accuracy on lumpy skin disease specifically looks weak, two more public
Kaggle datasets can be merged in the same way to add more examples:
- `warcoder/lumpy-skin-images-dataset`
- `shivamagarwal29/cow-lumpy-disease-dataset`

Both are lumpy-skin-vs-healthy only (no foot-and-mouth class), so they'd need
their images merged into this notebook's existing `lumpy skin disease` /
`healthy` folders before re-running the `image_dataset_from_directory` cell -
not required for v1.